__Лабораторная работа №20__

классификация тональности отзывов на фильмы с помощью предварительно обученной модели BERT

- Скачайте датасет отзывов на фильмы. Датасет содержит текст отзыва и бинарную метку тональности (положительный/отрицательный).
- Используйте библиотеку Hugging Face для загрузки предварительно обученной модели BERT и токенизатора.
- Подготовьте данные: используйте токенизатор BERT для преобразования текстовых данных в формат, который можно подать на вход модели BERT.
- Создайте классификатор на основе BERT: это может быть модель BERT с одним линейным слоем для классификации на вершине.
- Обучите классификатор на данных обучения и оцените его производительность на данных для тестирования.


In [1]:
import numpy as np
from numpy import random as rng
import re
import pandas as pd
from tqdm.auto import tqdm

pd.set_option('max_colwidth', 0)

# загружаем данные

In [2]:
# !ls -lh data/

In [3]:
%%time 

file_id='1RZJzx3nQlgmxKG-l3p7KldWBl_HVW3N9'
url= f'https://drive.google.com/uc?id={file_id}'
data =  pd.read_csv(url).convert_dtypes()
display(len(data))

assert ( len(data) > 40_000 )
# data.to_pickle('data/IMDB_dataset.pkl.gz',compression='gzip')

In [4]:
# data =  pd.read_csv('data/IMDB Dataset.csv').convert_dtypes()

In [5]:
display(len(data))
display(data.sample(9))

50000

review  \
6546   All the boys seem to be sexually aroused by Mandy Lane. All the girls seem to be jealous of Mandy Lane. But, nothing seems to become of it, and this viewer wonders why? Mandy is beautiful and a magnet to every boy she meets, but we never get to know Mandy or any of the characters in the film. Mandy accepts an invitation, from her student friends, to go to a secluded ranch. Three boys and three girls drink and drug. In the film, the teenagers drink booze like its water, and take drugs to experience a psychedelic trip. And, there is absolutely no sex. In the meantime, the teenagers disappear one by one. But, the others are all drunk and high. Nobody, including those watching the film, cares or is at all concerned. Nobody, including the audience, seems to give a damn. Emmet, a fellow student, is the instigator of the entire event. There is a security guard, Garth (dashingly and handsomely played by Anson Mount), who guards and protects the ranch. Midway through the film, the killer is revealed, the tension is suddenly released like air let out of a balloon. The events are completely predictable, and the film just completely fizzles out. Mandy meets her match, but we don't ever know why--and, at the end of the film, there is still no sex. Does Mandy hypnotize the boys, or does she simply bore all of the boys and girls to their deaths? This absolutely-confused viewer can only conclude that Mandy wishes to get rid of the female and male competition--by killing off the manipulative girls and the nasty boys.<br /><br />Is Mandy worth all of the attention? The director (Jonathan Levine) seems to think so, but this viewer does not. The able cinematographer (Darren Genet) provides some stunning images but, in fact, his focus seems to be on Garth, who is quite the stud. Not all of the boys love Mandy, or do they? If you want to be bored enough to find out how this film winds up, my advice is to sleep midway through film, until you see the temptress Mandy and Garth's bulging crotch. But, don't wait for anything to happen. Yep, you guessed it. Mandy remains a virgin, and there's still no sex. I rank this film a 3 out of 10, but not because of Mandy. Why? Because all of the girls love Garth, and all voyeuristic eyes seem to be on Garth in a compromising position. But unfortunately, girls and boys, this film never seems to get beyond a disappointing and incomplete sexual fantasy. Mandy goes to a secluded ranch, and nothing sexual ever happens. The audience is led to horror on a ranch--and cannot help, but wonder why?                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

## предварительная очистка

In [6]:
display(len(data))
data['review'] = (
    data['review']
    .apply( lambda s: re.sub(r'<[a-z]+ ?/?>','',s) ) # убрать html tag
    .apply( lambda s: re.sub(r'[\n\r\t]',' ',s) ) # убрать CR LS TAB
    .apply( lambda s: re.sub(r'https?://[0-9a-z._%/-]+',' URL ',s) ) # убрать url
    .apply( lambda s: re.sub('&amp;','&',s) ) # заменить &
    .apply( lambda s: re.sub(r'\s+',' ',s) ) # убрать повторы пробелов
    .str.strip() # убрать пробелы на концах
)
# убрать дубликаты
data = data.drop_duplicates(['review']).reset_index(drop=True)
display(len(data))

50000

49581

In [7]:
display(data.sample(9))

,review,sentiment
6166,"From the creators of Shrek .. OK, that grabbed my attention.Well the creators of Shrek also made Madagascar. Madagascar was half as good as Shrek.And now Flushed Away is half as good as Madagascar.That means Flushed Away isn't good. The animation and all that special effects were extremely good but the movie wasn't.The story of this movie was only meant for kids. It's seriously not possible for adults to actually love this flick.But there were many jokes meant for adults. I bet kids dint understand the jokes.Despite that I dint like this flick.I am completely disappointed. 4/10",negative
17218,"Great music, great dancing, a sexy star (Swayze) and actual sexual desire shown by a female character (a very unusual event!). You rarely see female characters who are motivated primarily by lust and it is a nice change. I also think the movie captures very well what it is like to be a teenager coping with issues of class and of having ideals suddenly come up against reality. Of course, it is sappy at moments - it is, after all, a movie musical - but all in all it is a real pleasure - a fun romance and also politically enlightened. This movie also has one of my favorite all time movie lines: ""I carried the watermelon.""",positive
1538,"The plot of The Thinner is decidedly thin. And gross. An obese lawyer drives over the Gypsy woman, and the Gypsy curse causes him to lose and lose weight... to the bone. OK, Gypsy curses should be entertaining, but the weight-losing gone bad? Nope. Except Stephen King thinks so. And Michael McDowell, other horror author and the screenwriter of this abysmal film, does so, too. The lawyer is not only criminally irresponsible, he is fat too, haha! The Thinner is like an immature piece of crap for a person who moans how he/she has never seen anything so disgusting than fatness. Hey, I can only say: Well, look at the mirror.",negative
21066,"I had seen 'Kalifornia' before (must be about 10 years ago) and I still remember to be very impressed by it. That's why I wanted to see it again and all I can say is that it still hasn't lost its power, even though I'm used to a lot more when it comes to movies than that I was ten years ago.'Kalifornia' tells the tale of the writer Brian Kessler and his girlfriend Carrie Laughlin, a photographer, who want to move to California. But instead of stepping on a plain and flying right to the state where they say it never rains, they choose to make a trip by car. He wants to write a book about America's most famous serial killers and she will make the matching pictures. But because their car uses an enormous amount of petrol, they decide to take another couple with them, so they can spread the costs of the trip. Only one couple has answered the add, so they will automatically be the lucky ones. But they haven't met each other yet and when seeing the other couple for the first time, when their trip has already started, Carrie is shocked. Without wanting to be prejudiced, she can only conclude that Early Grayce and Adele Corners are poor white trailer park trash. She definitely doesn't want them in her car, but Brian doesn't really mind to take them with them and decides to stop and pick them up anyway. At first the couple doesn't seem to be that bad after all, but gradually Early Grayce changes from a trashy hillbilly into a remorseless murderer...Not only is the story very impressive, so is the acting from our four leads. Brad Pitt is incredible as Early Grayce. His performance in this movie may well be his best ever. The same for Juliette Lewis. She plays the childish and naive girlfriend that doesn't want to hear a bad word about her Early and does that really very well. But David Duchovny and Michelle Forbes are a surprise as well. They both did a very good job and I really wonder why we never heard anything from Forbes again since this movie, because she really proves to have a lot of talent.Overall this is a very good and impressive psychological thr

In [8]:
# оценка размера текстов
data['review'].str.len().to_frame().describe([.01,.1,.25,.5,.75,.95,.99]).astype(int).T

,count,mean,std,min,1%,10%,25%,50%,75%,95%,99%,max
review,49581,1286,972,32,230,491,689,954,1560,3330,5099,13584


In [9]:
# замена текстовой метки на числовое значение
labels = { l:i for i,l in enumerate( sorted(set(data['sentiment'].dropna())) ) } 
display(labels)
data['target'] = data['sentiment'].map(labels)
display(data.sample(2))

{'negative': 0, 'positive': 1}

,review,sentiment,target
40310,"Let me first state that I rarely review movies, I only comment if I'm blown away or disappointed in something that I thought was going to be good. Killshot was a major disappointment on so many levels. The script was horrible, the acting was sub-par (espically coming from heavy weights like Rourke and Lane) and the editing and effects were comical, (blowing up cars etc. etc.) Rosario Dawson had a horrible role, I can't believe would even accept it, it was such a misuse of her talent I can't even put into words. I should have know after I saw the trailer for this movie 3 years ago and it kept being put on the shelve that their was a serious problem with this film.......... B movie all the way.........don't bother unless your really bored........",negative,0
41429,"What was this about ?? Pre-destination, you can not change the future cause it has already been written ??I'll give it this much. I did want to see what happened next and therefore watched the whole movie. This movie took a concept and made it watchable.If you're looking for a recommendation, See it at matinee prices. No thrills but an interesting concept. They should have left the Y2K reference out....",negative,0


## разделяем на учебный и тестовый набор

In [10]:
data = data.sample( 10_000 ) # для ускорения отладки сократим количество данных

In [11]:
train_size=0.8 # доля учебных примеров

data_train = data[['review','target']].sample( int(len(data)*train_size))
data_test = data[ ~data.index.isin(data_train.index) ]
data_train, data_test = data_train.reset_index(drop=True), data_test.reset_index(drop=True)
display( ( len(data_train),  len(data_test), ) )
# проверка - учебный и тестовый не пересекаются 
assert len( data_train[['review']].merge(data_test[['review']],how='inner') )== 0

(8000, 2000)

In [12]:
val_size=0.2 # доля контрольных примеров

data_train = data[['review','target']].sample( int(len(data)*(1.-val_size)))
data_val = data[ ~data.index.isin(data_train.index) ]
data_train, data_val = data_train.reset_index(drop=True), data_val.reset_index(drop=True)
display( ( len(data_train),  len(data_val), ) )
# проверка - учебный и контрольный не пересекаются 
assert len( data_train[['review']].merge(data_val[['review']],how='inner') )== 0

(8000, 2000)

In [13]:
# оценка сбалансированности данных (доля позитивных примеров)
display( data_train[['target']].describe().astype(int).T )
display( data_test[['target']].describe().astype(int).T )
display( data_val[['target']].describe().astype(int).T )

,count,mean,std,min,25%,50%,75%,max
target,8000,0,0,0,0,0,1,1


,count,mean,std,min,25%,50%,75%,max
target,2000,0,0,0,0,1,1,1


,count,mean,std,min,25%,50%,75%,max
target,2000,0,0,0,0,0,1,1


In [14]:
del data

# собираем генератор датасета

In [15]:
# !pip install transformers

In [16]:
import torch
from transformers import BertTokenizer

# Load the BERT tokenizer
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

In [17]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class TextDataset(Dataset):

    def __init__(self, text, target):
        super(TextDataset, self).__init__()
        assert len(text)>1
        assert len(set(target))>1
        assert len(text)==len(target)
        
        self._tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)
        self._text = text.copy()
        self._target = target.copy() #[np.newaxis,:].T
        self._n_class = max(target)+1
        
    
    def  _tokenize(self, sent):
        # `encode_plus` will:
        #    (1) Tokenize the sentence
        #    (2) Add the `[CLS]` and `[SEP]` token to the start and end
        #    (3) Truncate/Pad sentence to max length
        #    (4) Map tokens to their IDs
        #    (5) Create attention mask
        #    (6) Return a dictionary of outputs
        encoded_sent = self._tokenizer.encode_plus(
            text=sent,  # Preprocess sentence
            add_special_tokens=True,        # Add `[CLS]` and `[SEP]`
            max_length=64,                  # Max length to truncate/pad
            pad_to_max_length=True,         # Pad sentence to max length
            #truncation=True,
            #padding=True,
            #return_tensors='pt',           # Return PyTorch tensor
            return_attention_mask=True,     # Return attention mask
            )
        return (
            torch.tensor(encoded_sent.get('input_ids')),
            torch.tensor(encoded_sent.get('attention_mask'))
        )        

    def __len__(self):
        return len(self._text)
    
    def __getitem__(self, idx):
        return (
            self._tokenize( self._text[idx] ), 
            torch.eye( self._n_class, dtype=torch.float32 )[self._target[idx]]
            # torch.tensor(self._target[idx], dtype=torch.int )
            
        )

In [18]:
# Truncation was not explicitly activated but `max_length` is provided a specific value, 
# please use `truncation=True` to explicitly truncate examples to max length. 
# Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) 
# with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.

# /opt/venv/nb_3.11/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2377: 
# FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version,
# use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, 
# or use `padding='max_length'` to pad to a max length.
# In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) 
# or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).


In [19]:
# ds_train = TextDataset(text=data_train['review'].values,target=data_train['target'].values)
# for (input_tensor, attention_mask), target_tensor in DataLoader( ds_train, batch_size=3, shuffle=True): break
# display( input_tensor, attention_mask, target_tensor )

# собираем модель классификатора

In [20]:
%%time

import torch.nn as nn

class BertClassifier(nn.Module):

    def __init__(self, model_bert, out_size=2, hidden_size=128, ):
        super(BertClassifier, self).__init__()
        self._bert = model_bert # Instantiate BERT model
        bert_size = model_bert.pooler.dense.get_parameter('weight').shape[1]
        self._classifier = nn.Sequential(
            nn.Linear(bert_size, hidden_size), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden_size, out_size), nn.Softmax(dim=1),
            # nn.Linear(bert_size, out_size), nn.Softmax(dim=1),
        )

    def forward(self, input_ids, attention_mask):
        o = self._bert(input_ids=input_ids, attention_mask=attention_mask) # Feed input to BERT 
        o = o[0][:,0,:] # extract the last hidden state of the token `[CLS]` for classification task
        o = self._classifier(o) # feed input to classifier
        return o

CPU times: user 22 µs, sys: 1 µs, total: 23 µs
Wall time: 25.5 µs


In [21]:
# ds_train = TextDataset(text=data_train['review'].values,target=data_train['target'].values)
# for (input_tensor, attention_mask), target_tensor in DataLoader( ds_train, batch_size=3, shuffle=True): break
# display((input_tensor.shape, attention_mask.shape, target_tensor.shape,))

# model = BertClassifier()
# with torch.set_grad_enabled(False):
#     o = model(input_ids=input_tensor,attention_mask=attention_mask)
    
# display(o.shape)

---

In [22]:
# # проверяем GPU
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
# device='cpu'
display( device )

device(type='cuda', index=0)

In [23]:
from transformers import BertModel

model_bert = BertModel.from_pretrained('bert-base-uncased')
for param in model_bert.parameters():
    param.requires_grad = False  
    
model = BertClassifier(model_bert=model_bert.to(device)).to(device)    

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [24]:
from transformers import AdamW

optimizer = AdamW(model.parameters(), lr=5e-3 ) # ,  eps=1e-8  )
criterion = nn.CrossEntropyLoss()

2023-08-14 18:03:21.389230: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-08-14 18:03:22.040200: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/opt/venv/nb_3.11/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


## обучаем классификатор

In [25]:
# история значений ф-ции потери
loss_history_train = [] 
loss_history_val = [] 

In [26]:
# from torch.utils.data import RandomSampler, SequentialSampler

ds_train = TextDataset( text=data_train['review'].values, target=data_train['target'].values )
ds_val = TextDataset( text=data_val['review'].values, target=data_val['target'].values )

# train_sampler = RandomSampler(ds_train)

In [27]:
model = model.train()

In [28]:
n_epoch = 32 # количество эпох обучения
batch_size = 1024+512 # размер батча

lag_val_check = 7 # глубина истории для проверки изменения контрольной потери
n_iter_no_check = int(n_epoch*0.33) # количесво начальных итераций без проверки контрольной потери
display(n_iter_no_check)
assert n_iter_no_check>lag_val_check

10

In [29]:
from transformers import get_linear_schedule_with_warmup

total_steps = len(data_train) * n_epoch
scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=0,
        num_training_steps=total_steps,
    )

In [ ]:
%%time

for i in tqdm(range(n_epoch)): 
    loss_batch = []
    
    # получаем батч учебных примеров
    for (x,a),t in DataLoader(ds_train,batch_size=batch_size, shuffle=True): 
        model.zero_grad()
        o = model(input_ids=x.to(device), attention_mask=a.to(device))
        loss = criterion( o, t.to(device) )
        loss_batch.append(loss.item()) # дополняем историю изменения значений ф-ции потери на батче
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0) # cliping to prevent "exploding gradients"
        optimizer.step()
        
    scheduler.step()
   
    loss_history_train.append( np.mean(loss_batch) ) # дополняем общую историю изменения значений ф-ции потери
    
    
    loss_batch = []
    with torch.set_grad_enabled(False):   
        score = [
            model(input_ids=x.to(device), attention_mask=a.to(device))
            for (x,a),t in DataLoader(ds_val,batch_size=batch_size, shuffle=True) 
        ]

        # получаем батч проверочных примеров
        for (x,a), t in DataLoader( ds_val, batch_size=batch_size, shuffle=False): 
            o = model(input_ids=x.to(device), attention_mask=a.to(device))
            loss = criterion( o, t.to(device) )
            loss_batch.append(loss.item()) # дополняем историю изменения значений ф-ции потери на батче

    loss_history_val.append( np.mean(loss_batch) ) # дополняем общую историю изменения значений ф-ции потери

    if i>n_iter_no_check: # если контрольная потеря растёт, то прерываем процесс обучения
        if ( loss_history_val[-lag_val_check]<loss_history_val[-1] ):
            print( 'validation loss value up, stoped\n' )
            break

  0%|          | 0/32 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/opt/venv/nb_3.11/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2377: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(
Truncation was not explicitly activated but `max_length` is provided a specific value, pleas

In [ ]:
from matplotlib import pyplot as plt

# история изменения значений ф-ции потери
plt.plot( loss_history_train, label='loss_train min=%.3f'%(min(loss_history_train)), c='r' )
plt.plot( loss_history_val,   label='loss_val min=%.3f'%(min(loss_history_val)),     c='c' )
plt.grid()
plt.title('история изменения значений ф-ции потери')
plt.legend()

# тестируем модель

In [ ]:
# Put the model into the evaluation mode. 
# The dropout layers are disabled during the test time.
model = model.eval()

## оценка на учебном датасете

In [ ]:
# предииктим метки для тестового датасета
with torch.set_grad_enabled(False):
    score = np.vstack([
        model( x.to(device), a.to(device) ).cpu().numpy()
        for (x,a),t in tqdm( DataLoader(ds_train, batch_size=batch_size, shuffle=False) )
    ])  
    
target = data_train['target'].values       

In [ ]:
# рисуем ROC/AUC - зависимость точности и полноты при изменении порога скора
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions( target, score[:,1], name="class 1", color="darkorange", )
plt.plot([0,1],[0,1], color='navy', lw=1, linestyle='--')
plt.grid()

In [ ]:
# определяем оптимальный порог скора 
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve( target, score[:,1] )

#  максимальное количество True Positive при минимальном количестве False Positive
optimal_threshold = thresholds[ np.argmax( np.abs(tpr-fpr) )  ]
display(optimal_threshold)

# применяем оптимальный порог скора, предсказываем класс объектов
predicted = ( score[:,1] > optimal_threshold).astype(np.uint8)

In [ ]:
# оценка результатов тестирования

from sklearn.metrics import classification_report
print(classification_report(target,predicted))

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay( confusion_matrix=confusion_matrix(target,predicted), ).plot()

## оценка на контрольном датасете

In [ ]:
# предииктим метки для тестового датасета
with torch.set_grad_enabled(False):
    score = np.vstack([
        model( x.to(device), a.to(device) ).cpu().numpy()
        for (x,a),t in tqdm( DataLoader(ds_val, batch_size=batch_size, shuffle=False) )
    ])  
    
target = data_val['target'].values       

In [ ]:
# рисуем ROC/AUC - зависимость точности и полноты при изменении порога скора
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions( target, score[:,1], name="class 1", color="darkorange", )
plt.plot([0,1],[0,1], color='navy', lw=1, linestyle='--')
plt.grid()

In [ ]:
# определяем оптимальный порог скора 
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve( target, score[:,1] )

#  максимальное количество True Positive при минимальном количестве False Positive
optimal_threshold = thresholds[ np.argmax( np.abs(tpr-fpr) )  ]
display(optimal_threshold)

# применяем оптимальный порог скора, предсказываем класс объектов
predicted = ( score[:,1] > optimal_threshold).astype(np.uint8)

In [ ]:
# оценка результатов тестирования

from sklearn.metrics import classification_report
print(classification_report(target,predicted))

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay( confusion_matrix=confusion_matrix(target,predicted), ).plot()

## оценка на тестовом датасете

In [ ]:
# model = model.eval() 

In [ ]:
ds_test = TextDataset( text=data_test['review'].values, target=data_test['target'].values )

# предииктим метки для тестового датасета
with torch.set_grad_enabled(False):
    score = np.vstack([
        model( x.to(device), a.to(device) ).cpu().numpy()
        for (x,a),t in tqdm( DataLoader(ds_test, batch_size=batch_size, shuffle=False) )
    ])  
    
target = data_test['target'].values       

In [ ]:
# рисуем ROC/AUC - зависимость точности и полноты при изменении порога скора
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions( target, score[:,1], name="class 1", color="darkorange", )
plt.plot([0,1],[0,1], color='navy', lw=1, linestyle='--')
plt.grid()

In [ ]:
# определяем оптимальный порог скора 
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve( target, score[:,1] )

#  максимальное количество True Positive при минимальном количестве False Positive
optimal_threshold = thresholds[ np.argmax( np.abs(tpr-fpr) )  ]
display(optimal_threshold)

# применяем оптимальный порог скора, предсказываем класс объектов
predicted = ( score[:,1] > optimal_threshold).astype(np.uint8)

In [ ]:
# оценка результатов тестирования

from sklearn.metrics import classification_report
print(classification_report(target,predicted))

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay( confusion_matrix=confusion_matrix(target,predicted), ).plot()